# Module Transform — Nettoyage et structuration des données

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haja171106/donnee2-aqi/blob/feat/notebooks-analysis/notebooks/transform.ipynb)

Ce notebook analyse et documente le module `src/transform.py`.

**Rôle :** Lire tous les fichiers JSON bruts de `data/raw/`, les transformer en un unique fichier CSV propre dans `data/clean/qualite_air.csv`.

**Principes :**
- **Idempotence** : le résultat ne dépend que de `raw/`, pas du précédent `clean/`
- **Reconstruction totale** : le CSV est entièrement réécrit à chaque run
- **Déduplication** : une seule ligne par couple (ville, timestamp)

## 0. Configuration Colab

Cette cellule configure l'environnement que vous soyez dans Colab ou en local.

In [ ]:
import sys, os, json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    RAW_DIR = Path("/content/sample_data/raw")
    CLEAN_DIR = Path("/content/sample_data/clean")
    CLEAN_FILE = CLEAN_DIR / "qualite_air.csv"
    CLEAN_DIR.mkdir(parents=True, exist_ok=True)
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    import urllib.request
    print("Téléchargement du CSV depuis GitHub...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/haja171106/donnee2-aqi/feat/notebooks-analysis/data/clean/qualite_air.csv",
        CLEAN_FILE
    )
    print("CSV téléchargé ✓")
else:
    from config import RAW_DIR, CLEAN_FILE
    print("Configuration locale ✓")

COMPONENT_COLS = ["co", "no", "no2", "o3", "so2", "pm2_5", "pm10", "nh3"]
print("Configuration terminée ✓")

## 1. Analyse des fonctions

### `_rows_from_file(path) -> list[dict]`

Ouvre un fichier JSON brut et extrait chaque mesure horaire sous forme de dictionnaire.

**Étapes :**
1. Charge le fichier JSON
2. Récupère les métadonnées de la ville depuis `_city_meta`
3. Pour chaque élément de la clé `list` (une mesure par heure) :
   - Convertit le timestamp UNIX en ISO 8601
   - Extrait l'AQI (`main.aqi`)
   - Extrait les 8 polluants (`components.*`)
4. Retourne une liste de dictionnaires

In [ ]:
def _rows_from_file(path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    city_meta = payload.get("_city_meta", {})
    rows = []
    for item in payload.get("list", []):
        dt = datetime.fromtimestamp(item["dt"], tz=timezone.utc)
        row = {
            "ville": city_meta.get("name"),
            "pays": city_meta.get("country"),
            "latitude": city_meta.get("lat"),
            "longitude": city_meta.get("lon"),
            "timestamp_utc": dt.isoformat(),
            "aqi": item.get("main", {}).get("aqi"),
        }
        components = item.get("components", {})
        for col in COMPONENT_COLS:
            row[col] = components.get(col)
        rows.append(row)
    return rows

print("Fonction _rows_from_file définie ✓")

### `rebuild_clean() -> pd.DataFrame`

Fonction principale qui :
1. Parcourt **tous** les fichiers `*.json` de `data/raw/`
2. Extrait les lignes via `_rows_from_file`
3. Les assemble dans un DataFrame pandas
4. **Déduplique** sur (ville, timestamp_utc)
5. **Trie** chronologiquement par ville
6. **Écrit** `data/clean/qualite_air.csv`

**Pourquoi une reconstruction totale ?** Plutôt que d'ajouter les nouvelles lignes à la fin (append), on reconstruit tout depuis zéro. Cela garantit qu'aucune anomalie ne s'accumule entre les runs et simplifie la déduplication.

In [ ]:
def rebuild_clean() -> pd.DataFrame:
    all_rows = []
    for path in sorted(RAW_DIR.glob("*.json")):
        all_rows.extend(_rows_from_file(path))

    df = pd.DataFrame(all_rows)
    if df.empty:
        print("[transform] Aucune donnée brute trouvée — clean/ non modifié.")
        return df

    df = df.drop_duplicates(subset=["ville", "timestamp_utc"])
    df = df.sort_values(["ville", "timestamp_utc"]).reset_index(drop=True)
    df.to_csv(CLEAN_FILE, index=False)
    print(f"[transform] {len(df)} lignes écrites")
    return df

print("Fonction rebuild_clean définie ✓")

## 2. Analyse du fichier clean produit

Regardons la structure et les statistiques du fichier `qualite_air.csv`.

In [ ]:
df = pd.read_csv(CLEAN_FILE)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes\n")
print("Colonnes :")
for col in df.columns:
    print(f"  • {col}")
print(f"\nTypes :\n{df.dtypes}")

In [ ]:
print("Aperçu des 5 premières lignes :")
df.head(5)

In [ ]:
print("Statistiques descriptives :")
df.describe()

In [ ]:
print("Nombre de lignes par ville :")
df["ville"].value_counts()

In [ ]:
duplicates = df.duplicated(subset=["ville", "timestamp_utc"]).sum()
print(f"Doublons sur (ville, timestamp_utc) : {duplicates}")

tri_ok = True
for ville, grp in df.groupby("ville"):
    if not grp["timestamp_utc"].is_monotonic_increasing:
        print(f"  ⚠ {ville} : timestamps non triés")
        tri_ok = False
if tri_ok:
    print("✓ Toutes les villes ont leurs timestamps triés chronologiquement")

print(f"\nPériode : {df['timestamp_utc'].min()} → {df['timestamp_utc'].max()}")
print(f"Villes uniques : {df['ville'].nunique()}")
print(f"Polluants : {COMPONENT_COLS}")

## 3. Exemple : transformation d'un fichier brut

Chargeons un fichier brut et voyons la sortie de `_rows_from_file`.

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.json"))
if raw_files:
    rows = _rows_from_file(raw_files[0])
    df_sample = pd.DataFrame(rows)
    print(f"Fichier : {raw_files[0].name}")
    print(f"Lignes produites : {len(df_sample)}")
    df_sample
else:
    print("Aucun fichier raw trouvé — exécutez d'abord le notebook extract.ipynb")

## 4. Résumé

| Fonction | Rôle |
|---|---|
| `_rows_from_file(path)` | Extrait les mesures d'un fichier JSON brut en dictionnaires |
| `rebuild_clean()` | Parcourt raw/, déduplique, trie, écrit clean/qualite_air.csv |

**Points clés :**
- Reconstruction **totale** à chaque run (pas d'append)
- **Idempotent** : rejouable sans effet de bord
- Déduplication sur (ville, timestamp_utc)
- Tri chronologique par ville
- 8 polluants extraits : CO, NO, NO2, O3, SO2, PM2.5, PM10, NH3
- AQI sur l'échelle OpenWeather 1-5